# 워밍업 — 스무고개, 질문 하나의 가치 🎯

웹 챕터 **"스무고개 — 질문 하나의 가치"** 와 짝을 이루는 Python 실습입니다.
웹에서 게임으로 잡은 "감"(정보 = 줄어드는 불확실성)을, 여기서는 **직접 시뮬레이션**으로 확인합니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('sample')

## 1. 반반 전략을 코드로: 컴퓨터가 스무고개를 풀게 하기

웹에서 직접 했던 업다운 게임을 컴퓨터에게 시켜 봅니다.
전략은 하나, **항상 남은 후보의 가운데를 묻는다**(반반 질문).

In [ ]:
import numpy as np

rng = np.random.default_rng(42)  # 결과 재현을 위해 시드 고정

def half_and_half(secret, n=64, verbose=True):
    """1~n 에서 항상 가운데를 묻는 반반 전략. 질문 횟수를 돌려준다.

    후보가 1개 남으면 물어볼 필요가 없다 — 이미 정답을 아니까 게임 끝.
    (그래서 64개 후보는 어떤 경우에도 질문 6번이면 충분하다.)
    """
    lo, hi = 1, n
    questions = 0
    while lo < hi:
        guess = (lo + hi) // 2  # 남은 후보의 한가운데
        questions += 1
        before = hi - lo + 1
        if guess == secret:
            if verbose:
                print(f'{questions}번째 질문 "{guess}?" → 🎉 정답! (후보 {before} → 1)')
            return questions
        elif guess < secret:
            lo = guess + 1  # "UP" — 더 크다
        else:
            hi = guess - 1  # "DOWN" — 더 작다
        if verbose:
            print(f'{questions}번째 질문 "{guess}?" → {"UP" if lo > guess else "DOWN"} (후보 {before} → {hi - lo + 1})')
    if verbose:
        print(f'후보가 1개라 물어볼 것도 없이 정답은 {lo}! (총 질문 {questions}번)')
    return questions

secret = int(rng.integers(1, 65))
print(f'숨은 숫자: {secret}\n')
half_and_half(secret)

후보 수가 질문마다 **절반씩** 줄어드는 게 보이나요? 64 → 32 → 16 → 8 → 4 → 2 → 1.

## 2. 찍기 vs 반반: 어느 전략이 이길까

숨은 숫자를 1000번 바꿔 가며 두 전략의 **평균 질문 횟수**를 재 봅니다.
- **찍기**: 1인가요? 2인가요? … 하나씩 확인
- **반반**: 항상 남은 후보의 가운데를 질문

In [ ]:
def one_by_one(secret, n=64):
    """1부터 하나씩 찍는 전략. 질문 횟수를 돌려준다.

    마지막 하나가 남으면 물어볼 필요가 없으니, 최악은 n-1 번(= 63번).
    """
    return min(secret, n - 1)  # "1인가요?", "2인가요?", ... secret 번째에 정답

secrets = rng.integers(1, 65, size=1000)
picks = np.array([one_by_one(s) for s in secrets])
halves = np.array([half_and_half(s, verbose=False) for s in secrets])

print(f'찍기: 평균 {picks.mean():5.2f}번, 최악 {picks.max()}번')
print(f'반반: 평균 {halves.mean():5.2f}번, 최악 {halves.max()}번')

반반 전략은 **최악이라도 6번**. 찍기는 평균 32번 안팎, 최악 63번.
질문의 "품질"이 이만큼 차이를 만듭니다. 그래프로도 확인해 봅시다.

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

q = np.arange(0, 7)
remain_half = 64 / 2**q            # 반반: 질문마다 절반
remain_pick = 64 - q               # 찍기: 질문마다 1개씩 (운 나쁜 경우)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(q, remain_half, 'o-', color='#6366f1', label='반반 질문 (가운데 묻기)')
ax.plot(q, remain_pick, 's-', color='#f59e0b', label='하나씩 찍기 (운 나쁘면)')
ax.set_xlabel('질문 횟수')
ax.set_ylabel('남은 후보 수')
ax.set_title('질문 하나의 가치: 후보가 줄어드는 속도')
ax.grid(True, color='#eef2f7')
ax.legend()
plt.show()

## 3. "절반으로 몇 번 접나": 다음 장의 예고편

후보 수마다 반반 질문이 몇 번 필요한지 직접 세어 봅니다.
그리고 NumPy 의 `np.log2` 가 내놓는 값과 나란히 놓아 보세요. **정확히 같습니다.**
(이 함수의 정체는 1장에서 게임으로 배웁니다. 지금은 "접는 횟수를 세 주는 함수구나" 정도면 충분!)

In [ ]:
def folds(n):
    """n 개 후보를 1개로 만들 때까지 절반으로 접는 횟수 (n 은 2의 거듭제곱)."""
    count = 0
    while n > 1:
        n //= 2
        count += 1
    return count

print('후보 수 | 접는 횟수 | np.log2')
print('-' * 30)
for n in [2, 4, 8, 16, 32, 64, 128, 1024]:
    print(f'{n:6d} | {folds(n):8d} | {np.log2(n):6.1f}')

## 4. 직접 풀어보기 ✏️

후보가 **1024개**라면 반반 전략으로 최대 몇 번 만에 정답을 확정할 수 있을까요?
(웹 챕터의 어려움 문제와 같은 원리예요. 위 표에 힌트가 있어요.)

In [ ]:
# TODO: 아래 answer 를 여러분의 답으로 바꾸고 실행하세요.
# 힌트: folds(1024) 또는 np.log2(1024)

answer = folds(1024)
print('내 답:', answer)
assert answer == 10, '다시 생각해 보세요 — 1024 를 절반으로 몇 번 접어야 1이 될까요?'
print('✅ 잘했어요! 후보가 1024개나 되어도 좋은 질문 10번이면 끝납니다.')

---
**오늘의 한 줄**: 정보 = 줄어드는 불확실성. 좋은 질문 = 후보를 절반으로 가르는 질문.

다음 단계: 1장부터 각 웹 챕터마다 이런 노트북이 하나씩 따라옵니다.
1장에서는 "접는 횟수"에 이름(log₂)과 단위(**비트**)를 붙입니다. 🙂